___
<h1> Machine Learning </h1>
<h2> Systems Engineering and Computer Technologies / Engenharia de Sistemas e Tecnologias Informáticas
(LESTI)</h2>
<h3> Instituto Superior de Engenharia / Universidade do Algarve </h3>

[LESTI](https://ise.ualg.pt/curso/1941) / [ISE](https://ise.ualg.pt) / [UAlg](https://www.ualg.pt)

Pedro J. S. Cardoso (pcardoso@ualg.pt)

___

# Hand-tuning a decision tree on digits

Fine-tune a `DecisionTreeClassifier` **by hand** on the digits dataset. Do not use grid search.

Use only this estimator:

https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html

Change **three** parameters, one at a time:

- `max_depth` — how deep the tree may grow. `None` means no limit.
- `min_samples_leaf` — minimum number of samples required in a leaf.
- `criterion` — split quality, `"gini"` or `"entropy"`.

The point is the **metrics**, not the search. For every setting, compare the training set with the held-out test set using accuracy, precision, recall, F1 (macro average), and the confusion matrix.

Digits has 10 classes and they are roughly balanced, so accuracy and macro-F1 stay close. What moves is the gap between train and test, and which digits get confused.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data,
    digits.target,
    test_size=0.25,
    stratify=digits.target,
    random_state=1,
)
print(X_train.shape, X_test.shape)

## Exercise 1 — baseline

Fit a tree with the default parameters (`random_state=1`). Fill in the four metrics for **both** the training set and the test set. Then plot the test confusion matrix.

Use `average="macro"` for precision, recall, and F1, so each digit counts the same.

In [ ]:
def metrics_row(model, X, y):
    pred = model.predict(X)
    return {
        "accuracy": accuracy_score(y, pred),  # TODO: already filled; check the others
        "precision": ??,  # TODO precision_score, average="macro"
        "recall": ??,     # TODO recall_score, average="macro"
        "f1": ??,         # TODO f1_score, average="macro"
    }

baseline = DecisionTreeClassifier(random_state=1)
baseline.fit(X_train, y_train)

print("train", metrics_row(baseline, X_train, y_train))
print("test ", metrics_row(baseline, X_test, y_test))

In [ ]:
# Test confusion matrix of the default tree
pred_test = baseline.predict(X_test)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_test),
    display_labels=digits.target_names,
).plot(cmap="Blues")
plt.show()

# Which digits are confused with each other?
...

## Exercise 2 — `max_depth`

Keep every other argument at its default. Fit one tree for each depth below and store the train and test metrics.

Depths: `1`, `3`, `5`, `10`, `None`.

Then answer, from the numbers:

1. At which depth does the **training** accuracy reach 1.0?
2. Does the **test** F1 keep rising after that, or does it stall?
3. Plot the test confusion matrix for depth `3` and for `None`. Which digits improve, and which stay confused?

In [ ]:
depths = [1, 3, 5, 10, None]

for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=1)
    model.fit(X_train, y_train)
    train_m = metrics_row(model, X_train, y_train)
    test_m = metrics_row(model, X_test, y_test)
    print(f"max_depth={depth!s:>4}  train F1={train_m['f1']:.3f}  test F1={test_m['f1']:.3f}  "
          f"train acc={train_m['accuracy']:.3f}  test acc={test_m['accuracy']:.3f}")

In [ ]:
# Confusion matrices: shallow tree vs unlimited tree
...

## Exercise 3 — `min_samples_leaf`

Fix `max_depth=None` and vary `min_samples_leaf` over `1`, `5`, `20`, `50`.

1. What happens to the **train** F1 as the leaf grows?
2. Which value gives the best **test** F1?
3. For that value, is precision higher than recall, or the other way around? On a balanced problem the two macro scores should be close. If they are not, look at the confusion matrix and say which digits are driving the gap.

In [ ]:
leaves = [1, 5, 20, 50]

for leaf in leaves:
    model = ??  # TODO DecisionTreeClassifier(min_samples_leaf=..., random_state=1)
    model.fit(X_train, y_train)
    train_m = metrics_row(model, X_train, y_train)
    test_m = metrics_row(model, X_test, y_test)
    print(
        f"min_samples_leaf={leaf:>2}  "
        f"train P/R/F1={train_m['precision']:.3f}/{train_m['recall']:.3f}/{train_m['f1']:.3f}  "
        f"test P/R/F1={test_m['precision']:.3f}/{test_m['recall']:.3f}/{test_m['f1']:.3f}"
    )

## Exercise 4 — `criterion`

Compare `criterion="gini"` and `criterion="entropy"`.
Use the `max_depth` and `min_samples_leaf` that looked best on the **test** F1 in the previous exercises. Keep `random_state=1`.

1. How large is the difference in test accuracy and in test F1?
2. Does the confusion matrix change which pair of digits is mixed up, or only the counts?
3. Which single setting would you keep, and **which metric** made you choose it? Say why you would not choose from the training score alone.

In [ ]:
for criterion in ("gini", "entropy"):
    model = DecisionTreeClassifier(
        criterion=criterion,
        max_depth=??,          # TODO: your choice from exercise 2
        min_samples_leaf=??,   # TODO: your choice from exercise 3
        random_state=1,
    )
    model.fit(X_train, y_train)
    test_m = metrics_row(model, X_test, y_test)
    print(criterion, test_m)
    # TODO: plot the test confusion matrix for this criterion

## Exercise 5 — mix the three parameters

Tuning one parameter at a time can miss a good combination. Fit every mix of the values you already tried:

- `criterion`: `"gini"`, `"entropy"`
- `max_depth`: `1`, `3`, `5`, `10`, `None`
- `min_samples_leaf`: `1`, `5`, `20`, `50`

Still by hand: nested loops, no `GridSearchCV`. Keep `random_state=1`. For each mix, record train and test accuracy, precision, recall, and F1.

1. Which combination has the best **test** F1?
2. Is that the same combination you would have kept after exercises 2–4?
3. Among the top test-F1 settings, which one has the smaller gap between train F1 and test F1?
4. Plot the test confusion matrix of the best mix. Did any digit pair that was confused in the default tree disappear?

In [ ]:
# Refit the best mix and plot its test confusion matrix.
# Compare it with the default tree from exercise 1.

# TODO: the parameter with the highest test F1
best_criterion = ??  
best_depth = ??
best_min_samples = ??

best_model = DecisionTreeClassifier(
    criterion=best_criterion,
    max_depth=best_depth,
    min_samples_leaf=best_min_samples,
    random_state=1,
)
best_model.fit(X_train, y_train)

pred_test = best_model.predict(X_test)
ConfusionMatrixDisplay(
    confusion_matrix(y_test, pred_test),
    display_labels=digits.target_names,
).plot(cmap="Blues")
plt.show()